# ML-03 — Frame My Lane as a Ranked Refresh Queue

This lane is a scoring/ranking problem, not a prediction-for-its-own-sake problem. One row is one pseudonymized content item, and the model's job is to rank pages for editorial review and refresh work. The output is a priority score that helps an editor decide which items should be inspected first.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

candidates = [
    Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
    Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
]
data_path = next((candidate for candidate in candidates if candidate.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

starter = pd.read_csv(data_path)

print(f"Loaded: {data_path.name}")
print(f"Rows: {len(starter):,}")
print(f"Unique content items: {starter['content_id'].nunique():,}")
print(f"Unique clients: {starter['client_id'].nunique():,}")

I would predict a decline-risk label for each content item: `is_declining_label = 1` when the later-window trend is down, otherwise 0. That label is observed from behavior in the data pipeline, not invented by a manual rule. I would keep `trend_direction` out of the feature set and only use it here to explain the target.

In [ ]:
target_preview = starter.loc[
    :,
    ["content_id", "client_id", "content_type", "trend_direction"],
].copy()
target_preview["is_declining_label"] = target_preview["trend_direction"].str.lower().eq("down").astype(int)

display(target_preview.head(8))
print(target_preview["is_declining_label"].value_counts().rename("rows").to_frame())

I would judge the ranked queue with precision@50: among the top 50 pages the model sends to review, how many are truly declining? That metric matches the real action because editors can only inspect a small batch at a time. ROC-AUC is useful as a secondary sanity check, but precision@50 is the operational number here.

In [ ]:
decline_rate = target_preview["is_declining_label"].mean()
summary = pd.DataFrame(
    {
        "metric": ["decline_rate", "recommended_success_metric"],
        "value": [f"{decline_rate:.1%}", "precision@50"],
    }
)
display(summary)

One row = one pseudonymized content item. This is the unit of analysis the ranker will score. The dataframe below is the slice I would actually hand to the modeling loop: content identity for grouping only, plus the behavioral and freshness signals the model can learn from.

In [ ]:
unit_of_analysis = starter.loc[
    :,
    [
        "content_id",
        "client_id",
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "impressions_90d",
        "sessions_90d",
        "avg_position",
        "ctr",
        "engagement_rate",
        "scroll_rate",
        "word_count",
    ],
]
print("One row = one content item")
print(f"Rows: {len(unit_of_analysis):,}")
display(unit_of_analysis.head(8))

ML beats a fixed rule because the pages worth refreshing are not separated by one clean threshold. Content type changes the missingness pattern, freshness and position interact, and a page can be high-visibility yet still be drifting down. A rule like 'refresh everything older than X days' would waste effort; a ranked score can combine the signals and let the strongest cases rise to the top.

In [ ]:
ml_reason_view = (
    starter.assign(is_declining_label=starter["trend_direction"].str.lower().eq("down").astype(int))
    .groupby("content_type", as_index=False)
    .agg(
        rows=("content_id", "size"),
        decline_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
        median_position=("avg_position", "median"),
        median_days_since_update=("days_since_last_update", "median"),
    )
    .sort_values("decline_rate", ascending=False)
    .reset_index(drop=True)
 )

display(ml_reason_view)
print(
    "The same broad rule would ignore these differences, but a model can learn how freshness, visibility, "
    "and content type interact."
 )

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.